**This notebook prepares data for producing Figure 1 in `larvae_metamorphosis.R`.**

Run with

* Python 3.8
* pandas 2.0.3
* numpy 1.24.3

In [1]:
# Import standard libraries
import os
from datetime import date

import pandas as pd
import numpy as np

**Specify directories**

In [2]:
DATA_IN = 'datasets/raw'
DATA_OUT = 'datasets/processed'

# Load raw data

In [3]:
winter_highlow_df = pd.read_csv(
    os.path.join(DATA_IN, "winter_temperature_weekly_highlow.csv"), 
    index_col = False, 
    parse_dates = ['week_of']
)
winter_highlow_df.dtypes

week_of          datetime64[ns]
highlow_temps            object
dtype: object

# Data wrangling

**Generate endpoints between transitions between high and low winter temperatures**

In [4]:
# parse highlow_temps column into multiple columns
winter_highlow_df['high'] = winter_highlow_df['highlow_temps'].apply(
    lambda x: int(x[1])
)
winter_highlow_df['low'] = winter_highlow_df['highlow_temps'].apply(
    lambda x: int(x[4])
)
winter_highlow_df['high_date'] = winter_highlow_df.apply(
    lambda x: x.week_of + pd.DateOffset(days = x.high),
    axis = 1
)
winter_highlow_df['low_date'] = winter_highlow_df.apply(
    lambda x: x.week_of + pd.DateOffset(days = x.low),
    axis = 1
)
winter_highlow_df.dtypes

week_of          datetime64[ns]
highlow_temps            object
high                      int64
low                       int64
high_date        datetime64[ns]
low_date         datetime64[ns]
dtype: object

In [5]:
# helper function
def get_winter_tempofday(high: int, low: int, day: int) -> int: 
    ''' Returns the temperature for the given day
    
    Args
    ----
    high: day of week with high winter temperature as a number between 0 on 
    Monday and 6 on Sunday
    low: day of week with low winter temperature as a number between 0 on 
    Monday and 6 on Sunday
    day: day of week as a number between 0 on Monday and 6 on Sunday
    '''
    if high != day and low != day:
        return 10.5
    if high == day:
        return 14
    if low == day:
        return 7
    return -1

winter_highlow_df['day6_temp'] = winter_highlow_df.apply(
    lambda x: get_winter_tempofday(x.high, x.low, 6),
    axis = 1
)
winter_highlow_df['prev_temp'] = pd.concat(
    [pd.Series(np.nan), winter_highlow_df['day6_temp'][:-1]], 
    ignore_index = True
)
winter_highlow_df.tail(10)

,week_of,highlow_temps,high,low,high_date,low_date,day6_temp,prev_temp
8,2023-10-09,"[5, 6]",5,6,2023-10-14,2023-10-15,7.0,10.5
9,2023-10-16,"[5, 1]",5,1,2023-10-21,2023-10-17,10.5,7.0
10,2023-10-23,"[3, 4]",3,4,2023-10-26,2023-10-27,10.5,10.5
11,2023-10-30,"[0, 6]",0,6,2023-10-30,2023-11-05,7.0,10.5
12,2023-11-06,"[2, 1]",2,1,2023-11-08,2023-11-07,10.5,7.0
13,2023-11-13,"[6, 2]",6,2,2023-11-19,2023-11-15,14.0,10.5
14,2023-11-20,"[2, 0]",2,0,2023-11-22,2023-11-20,10.5,14.0
15,2023-11-27,"[1, 2]",1,2,2023-11-28,2023-11-29,10.5,10.5
16,2023-12-04,"[4, 6]",4,6,2023-12-08,2023-12-10,7.0,10.5
17,2023-12-11,"[2, 4]",2,4,2023-12-13,2023-12-15,10.5,7.0


In [6]:
def get_winter_temp_time_plot_pts(
    first_dateofweek, 
    high_date, 
    low_date, 
    high, 
    low,
    prev_temp
):
    ''' Returns a dataframe of endpoints between transitions in daily winter 
    temperatures for a given week
    
    Args
    ----
    first_dateofweek: first date of week in numpy datetime64[ns] format
    high_date: date of week with high winter temperature in numpy 
     datetime64[ns] format
    low_date: date of week with low winter temperature in numpy 
     datetime64[ns] format
    high: day of week with high winter temperature as a number between 0 on 
     Monday and 6 on Sunday
    low: day of week with low winter temperature as a number between 0 on 
     Monday and 6 on Sunday
    prev_temp: temperature on the last day of the previous week
    '''
    winter_hightemp_time_plot_pts_df = pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime(high_date.strftime('%Y-%m-%d') + " 09:00:00"), 
            pd.to_datetime(high_date.strftime(
                '%Y-%m-%d'
            ) + " 03:00:00") + pd.DateOffset(days = 1)
        ]), 
        'temperature': [14, 14], 
        'transition': ['n'] * 2
    })
    winter_lowtemp_time_plot_pts_df = pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime(low_date.strftime('%Y-%m-%d') + " 09:00:00"), 
            pd.to_datetime(low_date.strftime(
                '%Y-%m-%d'
            ) + " 03:00:00") + pd.DateOffset(days = 1)
        ]), 
        'temperature': [7, 7], 
        'transition': ['n'] * 2
    })
    winter_temp_time_plot_pts_df = pd.concat([
        winter_hightemp_time_plot_pts_df, 
        winter_lowtemp_time_plot_pts_df
    ], ignore_index = True)
    if abs(high - low) > 1:
        winter_betweentemp_time_plot_pts_df = pd.DataFrame({
            'datetime': pd.Series([
                pd.to_datetime(low_date.strftime(
                    '%Y-%m-%d'
                ) + " 09:00:00") + pd.DateOffset(days = 1), 
                pd.to_datetime(high_date.strftime(
                    '%Y-%m-%d'
                ) + " 03:00:00"), 
            ]), 
            'temperature': [10.5, 10.5], 
            'transition': ['n'] * 2
        })
        if low > high:
            winter_betweentemp_time_plot_pts_df = pd.DataFrame({
                'datetime': pd.Series([
                    pd.to_datetime(high_date.strftime(
                        '%Y-%m-%d'
                    ) + " 09:00:00") + pd.DateOffset(days = 1), 
                    pd.to_datetime(low_date.strftime(
                        '%Y-%m-%d'
                    ) + " 03:00:00"), 
                ]), 
                'temperature': [10.5, 10.5], 
                'transition': ['n'] * 2
            })
        winter_temp_time_plot_pts_df = pd.concat([
            winter_temp_time_plot_pts_df, 
            winter_betweentemp_time_plot_pts_df
        ], ignore_index = True)
    if high < 6 and high > low:
        winter_endtemp_time_plot_pts_df = pd.DataFrame({
            'datetime': pd.Series([pd.to_datetime(high_date.strftime(
                '%Y-%m-%d'
            ) + " 09:00:00") + pd.DateOffset(days = 1)]),
            'temperature': [10.5], 
            'transition': ['n']
        })
        winter_temp_time_plot_pts_df = pd.concat([
            winter_temp_time_plot_pts_df, 
            winter_endtemp_time_plot_pts_df
        ], ignore_index = True)
    elif low < 6 and high < low:
        winter_endtemp_time_plot_pts_df = pd.DataFrame({
            'datetime': pd.Series([pd.to_datetime(low_date.strftime(
                '%Y-%m-%d'
            ) + " 09:00:00") + pd.DateOffset(days = 1)]),
            'temperature': [10.5], 
            'transition': ['n']
        })
        winter_temp_time_plot_pts_df = pd.concat([
            winter_temp_time_plot_pts_df, 
            winter_endtemp_time_plot_pts_df
        ], ignore_index = True)
    if high < low and high > 0:
        winter_starttemp_time_plot_pts_df = pd.DataFrame({
            'datetime': pd.Series([pd.to_datetime(high_date.strftime(
                '%Y-%m-%d'
            ) + " 03:00:00")]),
            'temperature': [10.5], 
            'transition': ['n']
        })
        winter_temp_time_plot_pts_df = pd.concat([
            winter_temp_time_plot_pts_df, 
            winter_starttemp_time_plot_pts_df
        ], ignore_index = True)
    elif high > low and low > 0:
        winter_starttemp_time_plot_pts_df = pd.DataFrame({
            'datetime': pd.Series([pd.to_datetime(low_date.strftime(
                '%Y-%m-%d'
            ) + " 03:00:00")]),
            'temperature': [10.5], 
            'transition': ['n']
        })
        winter_temp_time_plot_pts_df = pd.concat([
            winter_temp_time_plot_pts_df, 
            winter_starttemp_time_plot_pts_df
        ], ignore_index = True)
    if (high == 0 or low == 0) and prev_temp == 10.5:
        winter_starttemp_time_plot_pts_df = pd.DataFrame({
            'datetime': pd.Series([pd.to_datetime(high_date.strftime(
                '%Y-%m-%d'
            ) + " 03:00:00")]),
            'temperature': [10.5], 
            'transition': ['n']
        })
        if low == 0:
            winter_starttemp_time_plot_pts_df = pd.DataFrame({
                'datetime': pd.Series([pd.to_datetime(low_date.strftime(
                    '%Y-%m-%d'
                ) + " 03:00:00")]),
                'temperature': [10.5], 
                'transition': ['n']
            })
        winter_temp_time_plot_pts_df = pd.concat([
            winter_temp_time_plot_pts_df, 
            winter_starttemp_time_plot_pts_df
        ], ignore_index = True)
    return winter_temp_time_plot_pts_df
    
temp_winter_df = pd.concat(list(winter_highlow_df.apply(
    lambda x: get_winter_temp_time_plot_pts(
        x.week_of, 
        x.high_date, 
        x.low_date, 
        x.high, 
        x.low, 
        x.prev_temp
    ),
    axis = 1
)), ignore_index = True)
temp_winter_df.sort_values(by = 'datetime', inplace = True)

In [7]:
temp_winter_df.shape

(119, 3)

In [8]:
temp_winter_df.head(15)

,datetime,temperature,transition
0,2023-08-14 09:00:00,14.0,n
1,2023-08-15 03:00:00,14.0,n
2,2023-08-15 09:00:00,7.0,n
3,2023-08-16 03:00:00,7.0,n
4,2023-08-16 09:00:00,10.5,n
11,2023-08-21 03:00:00,10.5,n
7,2023-08-21 09:00:00,7.0,n
8,2023-08-22 03:00:00,7.0,n
9,2023-08-22 09:00:00,10.5,n
10,2023-08-27 03:00:00,10.5,n


In [9]:
temp_winter_df.tail(10)

,datetime,temperature,transition
106,2023-12-10 09:00:00,7.0,n
107,2023-12-11 03:00:00,7.0,n
118,2023-12-13 03:00:00,10.5,n
111,2023-12-13 09:00:00,14.0,n
112,2023-12-14 03:00:00,14.0,n
115,2023-12-14 09:00:00,10.5,n
116,2023-12-15 03:00:00,10.5,n
113,2023-12-15 09:00:00,7.0,n
114,2023-12-16 03:00:00,7.0,n
117,2023-12-16 09:00:00,10.5,n


In [10]:
# remove extra times of winter that should shift to spring temperatures
temp_winter_df = temp_winter_df[
    temp_winter_df['datetime'] < pd.to_datetime("2023-12-13 09:00:00")
]
temp_winter_df.tail(10)

,datetime,temperature,transition
101,2023-11-30 03:00:00,7.0,n
102,2023-11-30 09:00:00,10.5,n
110,2023-12-08 03:00:00,10.5,n
104,2023-12-08 09:00:00,14.0,n
105,2023-12-09 03:00:00,14.0,n
108,2023-12-09 09:00:00,10.5,n
109,2023-12-10 03:00:00,10.5,n
106,2023-12-10 09:00:00,7.0,n
107,2023-12-11 03:00:00,7.0,n
118,2023-12-13 03:00:00,10.5,n


**Define functions to generate endpoints between transitions in daily spring & summer temperatures**

In [11]:
# helper functions
def get_daily_temp_springfall(datetimes):
    assert(all(
        isinstance(datetimes, np.datetime64)) for d in datetimes
    )
    return pd.DataFrame({
        'datetime': datetimes,
        'temperature': [18, 21, 21, 18] * int(len(datetimes) / 4), 
        'transition': ['n'] * len(datetimes)
    })
def get_daily_temp_summer(datetimes):
    assert(all(
        isinstance(datetimes, np.datetime64)) for d in datetimes
    )
    return pd.DataFrame({
        'datetime': datetimes,
        'temperature': [25.5, 32.5, 32.5, 25.5] * int(len(datetimes) / 4),
        'transition': ['n'] * len(datetimes)
    })

# Generate endpoints between transitions in daily and seasonal temperatures

**Early-breeding temperature sequence in sequence/treatment 2**

In [12]:
t2_temp_df = pd.concat([
    # first day
    pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime("2023-05-08 17:30:00"),
            pd.to_datetime("2023-05-08 22:00:00")
        ]), 
        'temperature': [21, 18], 
        'transition': ['n'] * 2
    }),
    get_daily_temp_springfall(
        pd.concat([
            pd.Series(pd.date_range(
                start = "2023-05-09 09:00:00", 
                end = "2023-06-12 09:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-05-09 13:00:00", 
                end = "2023-06-12 13:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-05-09 17:30:00", 
                end = "2023-06-12 17:30:00"
            )), 
            pd.Series(pd.date_range(
                start = "2023-05-09 22:00:00", 
                end = "2023-06-12 22:00:00"
            ))
        ]).sort_values()
    ), 
    # day with transition
    pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime("2023-06-13 09:00:00"),
            pd.to_datetime("2023-06-13 13:00:00"),
            pd.to_datetime("2023-06-13 17:30:00"),
            pd.to_datetime("2023-06-13 20:15:00")
        ]), 
        'temperature': [18, 21, 21, 25.5], 
        'transition': ['n', 'n', 'y', 'y']
    }),
    get_daily_temp_summer(
        pd.concat([
            pd.Series(pd.date_range(
                start = "2023-06-14 07:00:00", 
                end = "2023-10-12 07:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-06-14 12:00:00", 
                end = "2023-10-12 12:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-06-14 15:45:00", 
                end = "2023-10-12 15:45:00"
            )), 
            pd.Series(pd.date_range(
                start = "2023-06-14 20:15:00", 
                end = "2023-10-12 20:15:00"
            ))
        ]).sort_values()
    ), 
    # day with transition
    pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime("2023-10-13 07:00:00"),
            pd.to_datetime("2023-10-13 13:00:00"),
            pd.to_datetime("2023-10-13 17:30:00"),
            pd.to_datetime("2023-10-13 20:15:00")
        ]), 
        'temperature': [25.5, 21, 21, 18], 
        'transition': ['y', 'y', 'n', 'n']
    }),
    get_daily_temp_springfall(
        pd.concat([
            pd.Series(pd.date_range(
                start = "2023-10-14 09:00:00", 
                end = "2023-11-09 09:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-10-14 13:00:00", 
                end = "2023-11-09 13:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-10-14 17:30:00", 
                end = "2023-11-09 17:30:00"
            )), 
            pd.Series(pd.date_range(
                start = "2023-10-14 22:00:00", 
                end = "2023-11-09 22:00:00"
            ))
        ]).sort_values()
    )
], ignore_index = True)
t2_temp_df['sequence_number'] = 2

In [13]:
t2_temp_df.head()

,datetime,temperature,transition,sequence_number
0,2023-05-08 17:30:00,21.0,n,2
1,2023-05-08 22:00:00,18.0,n,2
2,2023-05-09 09:00:00,18.0,n,2
3,2023-05-09 13:00:00,21.0,n,2
4,2023-05-09 17:30:00,21.0,n,2


In [14]:
t2_temp_df.tail()

,datetime,temperature,transition,sequence_number
737,2023-11-08 22:00:00,18.0,n,2
738,2023-11-09 09:00:00,18.0,n,2
739,2023-11-09 13:00:00,21.0,n,2
740,2023-11-09 17:30:00,21.0,n,2
741,2023-11-09 22:00:00,18.0,n,2


**Early-breeding temperature sequence in sequence/treatment 1**

In [15]:
t1_temp_df = pd.concat([
    t2_temp_df,
    get_daily_temp_springfall(
        pd.concat([
            pd.Series(pd.date_range(
                start = "2023-11-10 09:00:00", 
                end = "2023-12-14 09:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-11-10 13:00:00", 
                end = "2023-12-14 13:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-11-10 17:30:00", 
                end = "2023-12-14 17:30:00"
            )), 
            pd.Series(pd.date_range(
                start = "2023-11-10 22:00:00", 
                end = "2023-12-14 22:00:00"
            ))
        ]).sort_values()
    )
], ignore_index = True)
t1_temp_df['sequence_number'] = 1

**Late-breeding temperature sequence in sequence/treatment 3**

In [16]:
t3_temp_df = pd.concat([
    # first day
    pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime("2023-05-08 15:45:00"),
            pd.to_datetime("2023-05-08 20:15:00")
        ]), 
        'temperature': [32.5, 25.5], 
        'transition': ['n', 'n']
    }),
    get_daily_temp_summer(
        pd.concat([
            pd.Series(pd.date_range(
                start = "2023-05-09 07:00:00", 
                end = "2023-06-12 07:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-05-09 12:00:00", 
                end = "2023-06-12 12:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-05-09 15:45:00", 
                end = "2023-06-12 15:45:00"
            )), 
            pd.Series(pd.date_range(
                start = "2023-05-09 20:15:00", 
                end = "2023-06-12 20:15:00"
            ))
        ]).sort_values()
    ), 
    # day with transition
    pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime("2023-06-13 07:00:00"),
            pd.to_datetime("2023-06-13 13:00:00"),
            pd.to_datetime("2023-06-13 17:30:00"),
            pd.to_datetime("2023-06-13 20:15:00")
        ]), 
        'temperature': [25.5, 21, 21, 18], 
        'transition': ['y', 'y', 'n', 'n']
    }),
    get_daily_temp_springfall(
        pd.concat([
            pd.Series(pd.date_range(
                start = "2023-06-14 09:00:00", 
                end = "2023-08-13 09:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-06-14 13:00:00", 
                end = "2023-08-13 13:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-06-14 17:30:00", 
                end = "2023-08-13 17:30:00"
            )), 
            pd.Series(pd.date_range(
                start = "2023-06-14 22:00:00", 
                end = "2023-08-13 22:00:00"
            ))
        ]).sort_values()
    ), 
    # day with transition
    pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime("2023-08-14 09:00:00"),
            pd.to_datetime("2023-08-14 09:00:00")
        ]), 
        'temperature': [18, 14], 
        'transition': ['y', 'y']
    }),
    temp_winter_df,
    # day with transition
    pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime("2023-12-13 09:00:00"),
            pd.to_datetime("2023-12-13 09:00:00")
        ]), 
        'temperature': [14, 18], 
        'transition': ['y', 'y']
    }),    
    get_daily_temp_springfall(
        pd.concat([
            pd.Series(pd.date_range(
                start = "2023-12-13 09:00:00", 
                end = "2024-02-13 09:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-12-13 13:00:00", 
                end = "2024-02-13 13:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2023-12-13 17:30:00", 
                end = "2024-02-13 17:30:00"
            )), 
            pd.Series(pd.date_range(
                start = "2023-12-13 22:00:00", 
                end = "2024-02-13 22:00:00"
            ))
        ]).sort_values()
    ),
    # day with transition
    pd.DataFrame({
        'datetime': pd.Series([
            pd.to_datetime("2024-02-14 09:00:00"),
            pd.to_datetime("2024-02-14 13:00:00"),
            pd.to_datetime("2024-02-14 17:30:00"),
            pd.to_datetime("2024-02-14 20:15:00")
        ]), 
        'temperature': [18, 21, 21, 25.5], 
        'transition': ['n', 'n', 'y', 'y']
    }),
    get_daily_temp_summer(
        pd.concat([
            pd.Series(pd.date_range(
                start = "2024-02-15 07:00:00", 
                end = "2024-06-14 07:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2024-02-15 12:00:00", 
                end = "2024-06-14 12:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2024-02-15 15:45:00", 
                end = "2024-06-14 15:45:00"
            )), 
            pd.Series(pd.date_range(
                start = "2024-02-15 20:15:00", 
                end = "2024-06-14 20:15:00"
            ))
        ]).sort_values()
    )    
], ignore_index = True)
t3_temp_df['sequence_number'] = 3

**Late-breeding temperature sequence in sequence/treatment 4**

In [17]:
t4_temp_df = pd.concat([
    t3_temp_df[t3_temp_df['datetime'] <= pd.to_datetime(
        "2024-02-14 20:15:00"
    )],
    get_daily_temp_summer(
        pd.concat([
            pd.Series(pd.date_range(
                start = "2024-02-15 07:00:00", 
                end = "2024-05-16 07:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2024-02-15 12:00:00", 
                end = "2024-05-16 12:00:00"
            )),
            pd.Series(pd.date_range(
                start = "2024-02-15 15:45:00", 
                end = "2024-05-16 15:45:00"
            )), 
            pd.Series(pd.date_range(
                start = "2024-02-15 20:15:00", 
                end = "2024-05-16 20:15:00"
            ))
        ]).sort_values()
    )
], ignore_index = True)
t4_temp_df['sequence_number'] = 4

**Consolidate sequences to write in a .csv for plotting in R**

In [18]:
sequence_schedule_temp_df = pd.concat([
    t1_temp_df, 
    t2_temp_df, 
    t3_temp_df, 
    t4_temp_df, 
])
sequence_schedule_temp_df.head()

,datetime,temperature,transition,sequence_number
0,2023-05-08 17:30:00,21.0,n,1
1,2023-05-08 22:00:00,18.0,n,1
2,2023-05-09 09:00:00,18.0,n,1
3,2023-05-09 13:00:00,21.0,n,1
4,2023-05-09 17:30:00,21.0,n,1


In [19]:
sequence_schedule_temp_df.to_csv(
    os.path.join(DATA_OUT, "sequence_temperature_to_plot.csv"),
    index = False
)

# Generate endpoints between transitions in seasonal photoperiods

**Early-breeding photoperiod sequence in sequence/treatment 1**

In [20]:
t1_photo_start_times_s = [
    pd.to_datetime("2023-05-08 20:00:00"), 
    pd.to_datetime("2023-06-21")
]
t1_photo_summer_times_s = [
    pd.to_datetime("2023-06-21"), 
    pd.to_datetime("2023-09-14")
]
t1_photo_fall_times_s = [
    pd.to_datetime("2023-09-14"), 
    pd.to_datetime("2023-12-07")
]
t1_photo_winter_times_s = [
    pd.to_datetime("2023-12-07"), 
    pd.to_datetime("2023-12-15")
]
t1_photo_df = pd.concat([
    # start
    pd.DataFrame({
        'datetime': t1_photo_start_times_s,
        'daylight_hours': [13] * 2
    }), 
    # summer
    pd.DataFrame({
        'datetime': t1_photo_summer_times_s, 
        'daylight_hours': [14] * 2
    }), 
    # fall
    pd.DataFrame({
        'datetime': t1_photo_fall_times_s, 
        'daylight_hours': [12] * 2
    }),
    # winter
    pd.DataFrame({
        'datetime': t1_photo_winter_times_s, 
        'daylight_hours': [10] * 2
    })
], ignore_index = True)
t1_photo_df['sequence_number'] = 1

**Early-breeding photoperiod sequence in sequence/treatment 3**

In [21]:
t3_photo_winter_times_s = [
    pd.to_datetime("2023-12-07"), 
    pd.to_datetime("2024-03-01")
]
t3_photo_spring_times_s = [
    pd.to_datetime("2024-03-01"), 
    pd.to_datetime("2024-05-24")
]
t3_photo_summer_times_s = [
    pd.to_datetime("2024-05-24"), 
    pd.to_datetime("2024-06-15")
]
t3_photo_df = pd.concat([
    t1_photo_df[t1_photo_df['daylight_hours'] != 10],
    pd.DataFrame({
        'datetime': t3_photo_winter_times_s, 
        'daylight_hours': [10] * 2
    }),
    pd.DataFrame({
        'datetime': t3_photo_spring_times_s, 
        'daylight_hours': [12] * 2
    }),
    pd.DataFrame({
        'datetime': t3_photo_summer_times_s, 
        'daylight_hours': [14] * 2
    }),
], ignore_index = True)
t3_photo_df['sequence_number'] = 3

**Late-breeding photoperiod sequence in sequence/treatment 2**

In [22]:
t2_photo_start_times_s = [
    pd.to_datetime("2023-05-08 20:00:00"), 
    pd.to_datetime("2023-06-28")
]
t2_photo_fall_times_s = [
    pd.to_datetime("2023-06-28"), 
    pd.to_datetime("2023-09-21")
]
t2_photo_winter_times_s = [
    pd.to_datetime("2023-09-21"), 
    pd.to_datetime("2023-11-10")
]
t2_photo_df = pd.concat([
    # start
    pd.DataFrame({
        'datetime': t2_photo_start_times_s,
        'daylight_hours': [13] * 2
    }), 
    # fall
    pd.DataFrame({
        'datetime': t2_photo_fall_times_s, 
        'daylight_hours': [12] * 2
    }), 
    # winter
    pd.DataFrame({
        'datetime': t2_photo_winter_times_s, 
        'daylight_hours': [10] * 2
    })
], ignore_index = True)
t2_photo_df['sequence_number'] = 2

**Late-breeding photoperiod sequence in sequence/treatment 4**

In [23]:
t4_photo_winter_times_s = [
    pd.to_datetime("2023-09-21"), 
    pd.to_datetime("2023-12-14")
]
t4_photo_spring_times_s = [
    pd.to_datetime("2023-12-14"), 
    pd.to_datetime("2024-03-08")
]
t4_photo_summer_times_s = [
    pd.to_datetime("2024-03-08"), 
    pd.to_datetime("2024-05-17")
]
t4_photo_df = pd.concat([
    t2_photo_df[t2_photo_df['daylight_hours'] != 10],
    # winter
    pd.DataFrame({
        'datetime': t4_photo_winter_times_s, 
        'daylight_hours': [10] * 2
    }),
    # spring
    pd.DataFrame({
        'datetime': t4_photo_spring_times_s, 
        'daylight_hours': [12] * 2
    }), 
    # summer
    pd.DataFrame({
        'datetime': t4_photo_summer_times_s, 
        'daylight_hours': [14] * 2
    })
], ignore_index = True)
t4_photo_df['sequence_number'] = 4

**Consolidate all sequences to write in a .csv for plotting in R**

In [24]:
sequence_schedule_photo_df = pd.concat([
    t1_photo_df, 
    t2_photo_df, 
    t3_photo_df, 
    t4_photo_df, 
])
sequence_schedule_photo_df.tail()

,datetime,daylight_hours,sequence_number
5,2023-12-14,10,4
6,2023-12-14,12,4
7,2024-03-08,12,4
8,2024-03-08,14,4
9,2024-05-17,14,4


In [25]:
sequence_schedule_photo_df.to_csv(
    os.path.join(DATA_OUT, "sequence_photoperiod_to_plot.csv"),
    index = False
)